In [22]:
%load_ext autoreload
%autoreload 2

# 0. 所有的组

In [2]:
import pandas as pd
import numpy as np
import vectorbt as vbt

In [ ]:
# 准备数据
market = pd.read_parquet("../data_ohlcv.parquet")
f_mev = pd.read_parquet("../factors_mev_new.parquet")
f_ohlcv = pd.read_parquet("../factors_ohlcv.parquet")
f_ohlcv = f_ohlcv.fillna(0)
target_returns = market['close'].pct_change().shift(-1) 
factors = pd.concat([f_mev, f_ohlcv], axis=1, join='outer')
# 清洗数据
factors = factors.replace([np.inf, -np.inf], np.nan)
factors = factors.fillna(0)
rolling_ic = factors.apply(lambda col: col.rolling(window=144).corr(target_returns))

In [19]:
group_names = {
    0: "Medium_Term_Momentum_Exhaustion",  # （有效）48周期的动量/资金流指标，IC为负，反映中线趋势的衰竭
    1: "Short_Term_Volume_Price_Consistency", # （有效）6-12周期的价量相关性，核心是判断量价配合的真伪
    2: "Neutral_Price_Ranking",            # 纯Rank类因子，IC近乎0，属于无效的截面位置指标
    3: "Short_Term_Money_Flow_Reversion",   # （有效）CMF/MFI/AVR 组合，极短线资金流向带来的反转信号
    4: "Short_Term_Price_Floor_Distance",   # （有效）价格距离6-12周期低点的位移，典型的超跌反转逻辑
    5: "Volatility_and_Realized_Risk",      # ATR/GKVol/RVol 组合，衡量波动率和风险分布
    6: "Volume_Surge_and_Delta_Inertia",    # 成交量突变与Delta（净主动买入）的微观特征
    7: "Medium_Term_Overbought_Oversold",   # （有效）48周期的ROC/CCI/DRP，反映中期维度的价格乖离
    8: "Medium_Term_Volatility_Profile",    # 48周期的波动率属性，属于长窗期的风险度量
    9: "Statistical_Tail_Risk_Features",    # 偏度(Skew)与峰度(Kurt)，刻画收益率分布的肥尾特征
    10: "Aggressive_Short_Term_Mean_Reversion", # （有效）核心组：6-12周期极短线反转，涵盖了最强的alpha来源
    11: "sandwich_intensity",
    12: "mev_vol_share",
    13: "attack_eff"
}

In [10]:
group_split = [['roc_48', 'mfi_48', 'avr_48', 'pvcorr_48', 'corr_48'],\
                ['pvcorr_6', 'pvcorr_12', 'corr_6', 'corr_12'], \
                ['rank_6', 'rank_12', 'rank_48'], \
                ['cmf_6', 'cmf_12', 'cmf_48', 'mfi_12', 'avr_6', 'avr_12'],\
                ['dist_l_6', 'dist_l_12'], \
                ['atr_6', 'atr_12', 'gkvol_6', 'gkvol_12', 'rvol_6', 'rvol_12'], \
                ['volsu_6', 'volsu_12', 'volsu_48', 'delta_6', 'delta_12', 'delta_48'],\
                ['roc_12', 'madev_48', 'cci_48', 'dist_h_48', 'dist_l_48', 'drp_48'], \
                ['atr_48', 'gkvol_48', 'rvol_48'], \
                ['skew_6', 'skew_12', 'skew_48', 'kurt_6', 'kurt_12', 'kurt_48'], \
                ['roc_6', 'madev_6', 'madev_12', 'cci_6', 'cci_12', 'dist_h_6', 'dist_h_12', 'mfi_6', 'drp_6', 'drp_12'], \
                ['f_sandwich_intensity_z'], \
                ['f_mev_vol_share_z'], \
                ['f_attack_eff_z']]
print(group_split)

[['roc_48', 'mfi_48', 'avr_48', 'pvcorr_48', 'corr_48'], ['pvcorr_6', 'pvcorr_12', 'corr_6', 'corr_12'], ['rank_6', 'rank_12', 'rank_48'], ['cmf_6', 'cmf_12', 'cmf_48', 'mfi_12', 'avr_6', 'avr_12'], ['dist_l_6', 'dist_l_12'], ['atr_6', 'atr_12', 'gkvol_6', 'gkvol_12', 'rvol_6', 'rvol_12'], ['volsu_6', 'volsu_12', 'volsu_48', 'delta_6', 'delta_12', 'delta_48'], ['roc_12', 'madev_48', 'cci_48', 'dist_h_48', 'dist_l_48', 'drp_48'], ['atr_48', 'gkvol_48', 'rvol_48'], ['skew_6', 'skew_12', 'skew_48', 'kurt_6', 'kurt_12', 'kurt_48'], ['roc_6', 'madev_6', 'madev_12', 'cci_6', 'cci_12', 'dist_h_6', 'dist_h_12', 'mfi_6', 'drp_6', 'drp_12'], ['f_sandwich_intensity_z'], ['f_mev_vol_share_z'], ['f_attack_eff_z']]


In [ ]:
group_eff = [0, 1, 3, 4, 7, 10]  # 有效组的索引

In [21]:
# 筛选一下组内平均因子ic
group_eff = []
for group_id in range(len(group_split)):
    print(f"Group {group_id}:")
    group_ic = []
    factor_list = group_split[group_id]
    for factor in factor_list:
        ic = factors[factor].corr(target_returns, method='spearman')
        group_ic.append(ic)
        print(f"  {factor}: {ic:.4f}")
    print(f"  Group {group_id} {group_names[group_id]} : Average IC = {np.mean(group_ic):.4f}")
    if np.abs(np.mean(group_ic)) > 0.01 : group_eff.append(group_id)
print(f"Effective Groups: {group_eff}")

Group 0:
  roc_48: -0.0248
  mfi_48: -0.0182
  avr_48: -0.0191
  pvcorr_48: -0.0130
  corr_48: -0.0075
  Group 0 Medium_Term_Momentum_Exhaustion : Average IC = -0.0165
Group 1:
  pvcorr_6: -0.0306
  pvcorr_12: -0.0260
  corr_6: -0.0242
  corr_12: -0.0208
  Group 1 Short_Term_Volume_Price_Consistency : Average IC = -0.0254
Group 2:
  rank_6: -0.0005
  rank_12: -0.0001
  rank_48: 0.0006
  Group 2 Neutral_Price_Ranking : Average IC = -0.0000
Group 3:
  cmf_6: -0.0444
  cmf_12: -0.0349
  cmf_48: -0.0170
  mfi_12: -0.0380
  avr_6: -0.0482
  avr_12: -0.0382
  Group 3 Short_Term_Money_Flow_Reversion : Average IC = -0.0368
Group 4:
  dist_l_6: -0.0419
  dist_l_12: -0.0406
  Group 4 Short_Term_Price_Floor_Distance : Average IC = -0.0412
Group 5:
  atr_6: -0.0019
  atr_12: -0.0024
  gkvol_6: 0.0044
  gkvol_12: 0.0035
  rvol_6: 0.0037
  rvol_12: 0.0032
  Group 5 Volatility_and_Realized_Risk : Average IC = 0.0017
Group 6:
  volsu_6: -0.0003
  volsu_12: 0.0005
  volsu_48: 0.0010
  delta_6: 0.0008
 

# 1. 第一组